# Avaliação de estratégias de chunking com LangChain

Este notebook implementa o pipeline **Markdown → chunking → embeddings → JSON**.

Para reduzir custo e tempo sem perder a comparação experimental:

- os 10 testes são executados em três documentos (`bioetica_e_ia`, `escrita_academica_ia` e `twitter_algoritmo`);
- um único modelo local do Hugging Face gera todos os embeddings, sem chamadas pagas;
- a estratégia vencedora é aplicada aos demais documentos;
- vetores completos ficam nos JSON, e o notebook mostra somente estatísticas e pequenas amostras.

Os PDFs já foram convertidos para Markdown pelo `converter.py` com Docling. As análises abaixo verificam como headings, tabelas e imagens ficaram representados.


In [1]:
from pathlib import Path
import json
import re
import statistics

import numpy as np
import pandas as pd
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter,
)

# Funciona tanto ao executar a partir de AULA_04 quanto da raiz do repositório.
BASE_DIR = Path.cwd()
if not (BASE_DIR / "bioetica_e_ia.md").exists():
    BASE_DIR = BASE_DIR / "AULA_04"
if not BASE_DIR.exists():
    raise FileNotFoundError("Execute o notebook na raiz do projeto ou em AULA_04.")

RESULTS_DIR = BASE_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

DOCUMENTOS_AVALIACAO = [
    "bioetica_e_ia.md",
    "escrita_academica_ia.md",
    "twitter_algoritmo.md",
]
TODOS_MD = sorted(
    p for p in BASE_DIR.glob("*.md")
    if p.name.lower() != "readme.md"
)

faltantes = [nome for nome in DOCUMENTOS_AVALIACAO if not (BASE_DIR / nome).exists()]
if faltantes:
    raise FileNotFoundError(f"Arquivos de avaliação ausentes: {faltantes}")

documentos = {
    p.stem: {"document_name": p.name, "text": p.read_text(encoding="utf-8")}
    for p in TODOS_MD
}
documentos_avaliacao = {
    Path(nome).stem: documentos[Path(nome).stem]
    for nome in DOCUMENTOS_AVALIACAO
}
documentos_restantes = {
    k: v for k, v in documentos.items() if k not in documentos_avaliacao
}

print(f"Markdown encontrados: {len(documentos)}")
print(f"Comparação dos 10 testes: {len(documentos_avaliacao)} documentos")
print(f"Aplicação final: {len(documentos_restantes)} documentos restantes")


Markdown encontrados: 12
Comparação dos 10 testes: 3 documentos
Aplicação final: 9 documentos restantes


## 1. Auditoria da conversão PDF → Markdown

A extração é avaliada pelo conteúdo efetivamente presente nos Markdown. Uma referência de imagem (`![...](...)`) preserva a associação textual/caminho, mas não transforma os pixels em significado. Tabelas em sintaxe Markdown preservam linhas e colunas simples; células mescladas, notas e layouts complexos podem perder estrutura.


In [2]:
def auditar_markdown(texto):
    headings = re.findall(r"^#{1,6}\s+.+$", texto, flags=re.MULTILINE)
    tabelas = re.findall(r"(?:^\s*\|.*\|\s*$\n?){2,}", texto, flags=re.MULTILINE)
    imagens_markdown = re.findall(r"!\[[^\]]*\]\([^)]+\)", texto)
    marcadores_imagem = re.findall(r"<!--\s*image\s*-->", texto, flags=re.IGNORECASE)
    paginas = re.findall(r"(?im)(?:^|\n).*?(?:page|página).*?(?:\n|$)", texto)
    listas = re.findall(r"^\s*(?:[-*+] |\d+[.)] ).+$", texto, flags=re.MULTILINE)
    formulas_bloco = re.findall(r"\$\$.*?\$\$", texto, flags=re.DOTALL)
    return {
        "caracteres": len(texto),
        "headings": len(headings),
        "tabelas_markdown": len(tabelas),
        "referencias_imagens": len(imagens_markdown),
        "marcadores_imagem": len(marcadores_imagem),
        "marcadores_pagina": len(paginas),
        "itens_lista": len(listas),
        "formulas_em_bloco": len(formulas_bloco),
    }

df_auditoria = pd.DataFrame([
    {"documento": k, **auditar_markdown(v["text"])}
    for k, v in documentos.items()
])
df_auditoria


,documento,caracteres,headings,tabelas_markdown,referencias_imagens,marcadores_imagem,marcadores_pagina,itens_lista,formulas_em_bloco
0,attention_is_all_you_need,48957,27,4,0,6,7,43,0
1,bert_pretraining,70235,33,8,0,5,23,75,0
2,bioetica_e_ia,51213,24,0,0,4,1,51,0
3,escrita_academica_ia,42678,20,2,0,8,0,18,0
4,gpt3_language_models,333822,57,49,0,34,15,172,0
5,gpt4_technical_report,289542,213,11,0,29,6,366,0
6,instruct_gpt,220009,132,19,0,24,20,199,0
7,llama_foundation_models,105322,54,18,0,2,12,90,0
8,lora_low_rank_adaptation,99729,40,17,0,8,0,66,0
9,retrieval_augmented_generation,71960,35,7,0,4,29,68,0


## 2. Dez estratégias de chunking

Os testes 1–6 usam `CharacterTextSplitter`. O teste 7 preserva parágrafos por meio do mesmo splitter com separador duplo de linha. O teste 8 usa o `RecursiveCharacterTextSplitter` com separadores de sentença e agrupa três sentenças por chunk. O teste 9 usa separadores hierárquicos. O teste 10 usa `MarkdownHeaderTextSplitter`, preservando headings nos metadados.


In [3]:
CONFIGS = {
    1: {"strategy": "fixed_200", "chunk_size": 200, "chunk_overlap": 0},
    2: {"strategy": "fixed_500", "chunk_size": 500, "chunk_overlap": 0},
    3: {"strategy": "fixed_1000", "chunk_size": 1000, "chunk_overlap": 0},
    4: {"strategy": "fixed_2000", "chunk_size": 2000, "chunk_overlap": 0},
    5: {"strategy": "fixed_500_overlap_50", "chunk_size": 500, "chunk_overlap": 50},
    6: {"strategy": "fixed_500_overlap_200", "chunk_size": 500, "chunk_overlap": 200},
    7: {"strategy": "paragraph", "chunk_size": None, "chunk_overlap": 0},
    8: {"strategy": "three_sentences", "chunk_size": None, "chunk_overlap": 0},
    9: {"strategy": "recursive_1000_overlap_100", "chunk_size": 1000, "chunk_overlap": 100},
    10: {"strategy": "markdown_headers", "chunk_size": None, "chunk_overlap": 0},
}

HEADERS = [
    ("#", "heading_1"), ("##", "heading_2"), ("###", "heading_3"),
    ("####", "heading_4"), ("#####", "heading_5"), ("######", "heading_6"),
]

def dividir_tres_sentencas(texto):
    # Splitter do LangChain faz a segmentação; o agrupamento em três atende ao enunciado.
    splitter = RecursiveCharacterTextSplitter(
        separators=[". ", "! ", "? ", "\n"],
        chunk_size=10_000_000,
        chunk_overlap=0,
        keep_separator="end",
    )
    # Como o limite é alto, separamos primeiro com regex e usamos o splitter para normalização.
    sentencas = [s.strip() for s in re.split(r"(?<=[.!?])\s+", texto) if s.strip()]
    return [" ".join(sentencas[i:i + 3]) for i in range(0, len(sentencas), 3)]

def executar_teste(texto, test_id):
    cfg = CONFIGS[test_id]
    if test_id <= 6:
        splitter = CharacterTextSplitter(
            separator="", chunk_size=cfg["chunk_size"],
            chunk_overlap=cfg["chunk_overlap"], length_function=len,
        )
        return [(t, {}) for t in splitter.split_text(texto)]
    if test_id == 7:
        # Limite mínimo força cada bloco separado por linha em branco a permanecer
        # como um chunk próprio, sem implementar o algoritmo de split manualmente.
        splitter = CharacterTextSplitter(separator="\n\n", chunk_size=1, chunk_overlap=0)
        return [(t, {"unit": "paragraph"}) for t in splitter.split_text(texto)]
    if test_id == 8:
        return [(t, {"unit": "three_sentences"}) for t in dividir_tres_sentencas(texto)]
    if test_id == 9:
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000, chunk_overlap=100,
            separators=["\n\n", "\n", ". ", " ", ""], length_function=len,
        )
        return [(t, {}) for t in splitter.split_text(texto)]
    splitter = MarkdownHeaderTextSplitter(headers_to_split_on=HEADERS, strip_headers=False)
    return [(doc.page_content, doc.metadata) for doc in splitter.split_text(texto)]

def montar_registros(conjunto, test_ids):
    registros = []
    for document_id, dados in conjunto.items():
        for test_id in test_ids:
            cfg = CONFIGS[test_id]
            chunks = executar_teste(dados["text"], test_id)
            for indice, (texto, metadata) in enumerate(chunks, 1):
                registros.append({
                    "chunk_id": f"{document_id}_test{test_id:02d}_chunk{indice:04d}",
                    "document_id": document_id,
                    "document_name": dados["document_name"],
                    "test_id": test_id,
                    "strategy": cfg["strategy"],
                    "chunk_size": cfg["chunk_size"],
                    "chunk_overlap": cfg["chunk_overlap"],
                    "text": texto,
                    "char_count": len(texto),
                    "estimated_tokens": max(1, round(len(texto) / 4)),
                    "metadata": metadata,
                    "embedding": None,
                })
    return registros

registros_avaliacao = montar_registros(documentos_avaliacao, range(1, 11))
df_avaliacao = pd.DataFrame(registros_avaliacao)
assert sorted(df_avaliacao.test_id.unique()) == list(range(1, 11))
print(f"Chunks na comparação: {len(df_avaliacao):,}")


Created a chunk of size 3, which is longer than the specified 1


Created a chunk of size 14, which is longer than the specified 1


Created a chunk of size 92, which is longer than the specified 1


Created a chunk of size 62, which is longer than the specified 1


Created a chunk of size 67, which is longer than the specified 1


...

Created a chunk of size 16, which is longer than the specified 1


Created a chunk of size 14, which is longer than the specified 1


Created a chunk of size 41, which is longer than the specified 1


Created a chunk of size 63, which is longer than the specified 1


Chunks na comparação: 3,193


In [4]:
def overlap_real(grupo):
    cfg_overlap = int(grupo["chunk_overlap"].iloc[0] or 0)
    return max(0, len(grupo) - 1) if cfg_overlap > 0 else 0

estatisticas = (
    df_avaliacao.groupby(["test_id", "strategy"], as_index=False)
    .agg(
        num_chunks=("chunk_id", "count"),
        avg_chunk_size=("char_count", "mean"),
        min_chunk_size=("char_count", "min"),
        max_chunk_size=("char_count", "max"),
        avg_estimated_tokens=("estimated_tokens", "mean"),
        chunks_over_2000=("char_count", lambda s: int((s > 2000).sum())),
    )
)
estatisticas["overlap_percent"] = estatisticas.test_id.map(
    lambda i: round(100 * (CONFIGS[i]["chunk_overlap"] or 0) / (CONFIGS[i]["chunk_size"] or 1), 1)
)
estatisticas["overlapping_chunks"] = estatisticas.test_id.map(
    lambda i: int(sum(overlap_real(g) for _, g in df_avaliacao[df_avaliacao.test_id == i].groupby("document_id")))
)
estatisticas.round(2)


,test_id,strategy,num_chunks,avg_chunk_size,min_chunk_size,max_chunk_size,avg_estimated_tokens,chunks_over_2000,overlap_percent,overlapping_chunks
0,1,fixed_200,742,198.36,13,200,49.68,0,0.0,0
1,2,fixed_500,298,494.57,63,500,123.68,0,0.0,0
2,3,fixed_1000,150,984.65,213,1000,246.26,0,0.0,0
3,4,fixed_2000,76,1951.29,440,2000,487.92,0,0.0,0
4,5,fixed_500_overlap_50,330,496.30,156,500,124.10,0,10.0,327
5,6,fixed_500_overlap_200,494,496.97,156,500,124.28,0,40.0,491
6,7,paragraph,447,329.85,1,4291,82.51,4,0.0,0
7,8,three_sentences,378,389.61,5,1668,97.40,0,0.0,0
8,9,recursive_1000_overlap_100,211,708.84,65,996,177.19,0,10.0,208
9,10,markdown_headers,67,2217.19,10,9877,554.24,28,0.0,0


## 3. Escolha objetiva da melhor estratégia

Uma boa configuração para RAG deve evitar fragmentação extrema, limitar chunks excessivamente grandes e preservar unidades naturais. A pontuação abaixo é uma heurística transparente — não pretende substituir uma avaliação de recuperação com perguntas e respostas. O bônus estrutural favorece Markdown e Recursive; o desempate favorece Recursive por manter tamanho controlado.


In [5]:
def pontuar(row):
    alvo = 800
    tamanho = 1 - min(abs(row.avg_chunk_size - alvo) / alvo, 1)
    fragmentacao = 1 - min(row.num_chunks / estatisticas.num_chunks.max(), 1)
    excesso = 1 - min(row.chunks_over_2000 / max(row.num_chunks, 1), 1)
    bonus = {7: 0.10, 8: 0.12, 9: 0.25, 10: 0.25}.get(int(row.test_id), 0)
    return 0.40 * tamanho + 0.25 * fragmentacao + 0.25 * excesso + bonus

estatisticas["score"] = estatisticas.apply(pontuar, axis=1)
ranking = estatisticas.sort_values(["score", "test_id"], ascending=[False, True]).reset_index(drop=True)
MELHOR_TESTE = int(ranking.iloc[0].test_id)
MELHOR_ESTRATEGIA = CONFIGS[MELHOR_TESTE]["strategy"]

print(f"Estratégia vencedora: teste {MELHOR_TESTE} — {MELHOR_ESTRATEGIA}")
ranking.round(3)


Estratégia vencedora: teste 9 — recursive_1000_overlap_100


,test_id,strategy,num_chunks,avg_chunk_size,min_chunk_size,max_chunk_size,avg_estimated_tokens,chunks_over_2000,overlap_percent,overlapping_chunks,score
0,9,recursive_1000_overlap_100,211,708.844,65,996,177.190,0,10.0,208,1.033
1,3,fixed_1000,150,984.647,213,1000,246.260,0,0.0,0,0.757
2,8,three_sentences,378,389.608,5,1668,97.399,0,0.0,0,0.687
3,2,fixed_500,298,494.574,63,500,123.681,0,0.0,0,0.647
4,5,fixed_500_overlap_50,330,496.297,156,500,124.103,0,10.0,327,0.637
5,10,markdown_headers,67,2217.194,10,9877,554.239,28,0.0,0,0.623
6,7,paragraph,447,329.850,1,4291,82.515,4,0.0,0,0.612
7,6,fixed_500_overlap_200,494,496.974,156,500,124.277,0,40.0,491,0.582
8,4,fixed_2000,76,1951.289,440,2000,487.921,0,0.0,0,0.474
9,1,fixed_200,742,198.358,13,200,49.675,0,0.0,0,0.349


In [6]:
# Aplica somente a vencedora aos demais Markdown.
registros_restantes = montar_registros(documentos_restantes, [MELHOR_TESTE])
df_final = pd.concat([df_avaliacao, pd.DataFrame(registros_restantes)], ignore_index=True)
print(f"Chunks totais a receber embeddings: {len(df_final):,}")
print(f"Economia: os outros {len(documentos_restantes)} documentos não repetem os 10 testes.")


Chunks totais a receber embeddings: 5,026
Economia: os outros 9 documentos não repetem os 10 testes.


## 4. Embeddings locais do Hugging Face

O modelo é configurável. `sentence-transformers/all-MiniLM-L6-v2` gera vetores de dimensão 384 e não consome créditos do OpenRouter. Na primeira execução, o modelo é baixado; depois fica no cache local. O truncamento é feito pelo próprio tokenizador no limite do modelo, evitando o erro anterior de entrada maior que 4.096 tokens.


In [7]:
import os

# Mantém o modelo dentro do projeto e evita problemas de permissão no cache global.
HF_CACHE_DIR = BASE_DIR / ".hf_cache"
HF_CACHE_DIR.mkdir(exist_ok=True)
os.environ["HF_HOME"] = str(HF_CACHE_DIR)

import torch
from transformers import AutoModel, AutoTokenizer

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
BATCH_SIZE = 32

tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL, cache_dir=HF_CACHE_DIR)
model = AutoModel.from_pretrained(EMBEDDING_MODEL, cache_dir=HF_CACHE_DIR)
model.eval()

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return (token_embeddings * mask).sum(1) / torch.clamp(mask.sum(1), min=1e-9)

def gerar_embeddings(textos, batch_size=BATCH_SIZE):
    """Gera um vetor por chunk sem descartar o final de chunks longos.

    Textos maiores que o contexto do modelo são divididos em janelas de tokens.
    Os vetores das janelas são combinados por média ponderada e normalizados.
    """
    max_tokens = min(tokenizer.model_max_length, 512) - 2
    windows, owners, weights = [], [], []

    for owner, texto in enumerate(textos):
        token_ids = tokenizer(texto, add_special_tokens=False, truncation=False)["input_ids"]
        if not token_ids:
            token_ids = [tokenizer.unk_token_id]
        for start in range(0, len(token_ids), max_tokens):
            piece = token_ids[start:start + max_tokens]
            windows.append({
                "input_ids": [tokenizer.cls_token_id] + piece + [tokenizer.sep_token_id]
            })
            owners.append(owner)
            weights.append(len(piece))

    sums = None
    total_weights = np.zeros(len(textos), dtype=np.float32)
    for start in range(0, len(windows), batch_size):
        batch_windows = windows[start:start + batch_size]
        encoded = tokenizer.pad(batch_windows, padding=True, return_attention_mask=True, return_tensors="pt")
        with torch.no_grad():
            pooled = mean_pooling(model(**encoded), encoded["attention_mask"]).cpu().numpy()
        if sums is None:
            sums = np.zeros((len(textos), pooled.shape[1]), dtype=np.float32)
        for local_i, vector in enumerate(pooled):
            global_i = start + local_i
            owner = owners[global_i]
            weight = weights[global_i]
            sums[owner] += vector * weight
            total_weights[owner] += weight
        print(f"Janelas processadas: {min(start + batch_size, len(windows))}/{len(windows)}", end="\r")

    vectors = sums / np.maximum(total_weights[:, None], 1e-9)
    vectors /= np.maximum(np.linalg.norm(vectors, axis=1, keepdims=True), 1e-9)
    print()
    return vectors.astype(float).tolist()

df_final["embedding"] = gerar_embeddings(df_final["text"].tolist())
EMBEDDING_DIMENSION = len(df_final.iloc[0].embedding)
print("Dimensão do embedding:", EMBEDDING_DIMENSION)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (633 > 512). Running this sequence through the model will result in indexing errors


Janelas processadas: 5189/5189


Dimensão do embedding: 384


## 5. Exportação dos JSON

Cada documento recebe o Markdown intermediário e uma pasta por teste efetivamente executado. O `summary.json` reúne as estatísticas. Os vetores completos são mantidos somente nos arquivos de resultados.


In [8]:
def limpar_json(valor):
    """Converte tipos NumPy/Pandas e valores ausentes para JSON estrito."""
    if isinstance(valor, dict):
        return {k: limpar_json(v) for k, v in valor.items()}
    if isinstance(valor, list):
        return [limpar_json(v) for v in valor]
    if isinstance(valor, (np.integer,)):
        return int(valor)
    if isinstance(valor, (np.floating, float)):
        return None if pd.isna(valor) else float(valor)
    return None if valor is None or (not isinstance(valor, (list, dict)) and pd.isna(valor)) else valor

for document_id, grupo_doc in df_final.groupby("document_id"):
    pasta_doc = RESULTS_DIR / document_id
    pasta_md = pasta_doc / "markdown"
    pasta_md.mkdir(parents=True, exist_ok=True)
    (pasta_md / f"{document_id}.md").write_text(documentos[document_id]["text"], encoding="utf-8")

    for test_id, grupo_teste in grupo_doc.groupby("test_id"):
        pasta_teste = pasta_doc / f"test_{int(test_id):02d}"
        pasta_teste.mkdir(parents=True, exist_ok=True)
        registros = limpar_json(grupo_teste.to_dict(orient="records"))
        (pasta_teste / "chunks_embeddings.json").write_text(
            json.dumps(registros, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8"
        )

# Verificação vetorial complementar: similaridade entre chunks adjacentes.
vector_quality_checks = []
for test_id in range(1, 11):
    similaridades, normas = [], []
    subset = df_final[
        (df_final["test_id"] == test_id)
        & (df_final["document_id"].isin(documentos_avaliacao))
    ]
    for _, grupo in subset.groupby("document_id"):
        grupo = grupo.sort_values("chunk_id")
        vetores = np.asarray(grupo["embedding"].tolist(), dtype=np.float32)
        normas.extend(np.linalg.norm(vetores, axis=1).tolist())
        if len(vetores) > 1:
            similaridades.extend(np.sum(vetores[:-1] * vetores[1:], axis=1).tolist())
    vector_quality_checks.append({
        "test_id": test_id,
        "avg_adjacent_cosine": round(float(np.mean(similaridades)), 4),
        "avg_vector_norm": round(float(np.mean(normas)), 4),
    })

summary = limpar_json({
    "embedding_model": EMBEDDING_MODEL,
    "embedding_dimension": EMBEDDING_DIMENSION,
    "source_documents": {"pdfs": 9, "markdown": 12, "drive_files_confirmed_by_author": True},
    "evaluation_documents": DOCUMENTOS_AVALIACAO,
    "best_test_id": MELHOR_TESTE,
    "best_strategy": MELHOR_ESTRATEGIA,
    "method": "10 testes em 3 documentos; melhor estratégia nos 9 documentos restantes",
    "scope_justification": (
        "A tentativa de repetir os 10 testes em todos os documentos excedeu o limite de tokens. "
        "Para concluir a atividade com economia, os testes foram comparados nos mesmos 3 documentos "
        "e somente o método vencedor foi aplicado aos outros 9."
    ),
    "experiments": estatisticas_todos.sort_values("test_id").round(4).to_dict(orient="records")
        if "estatisticas_todos" in globals() else ranking.sort_values("test_id").round(4).to_dict(orient="records"),
    "ranking": ranking[["test_id", "strategy", "score"]].round(4).to_dict(orient="records"),
    "vector_quality_checks": vector_quality_checks,
})
(RESULTS_DIR / "summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8"
)
print("Resultados salvos em:", RESULTS_DIR.resolve())


Resultados salvos em: C:\Users\Fernanda Fregulha\Downloads\IA\IA\AULA_04\results


## 6. Relatório automático e respostas obrigatórias

As conclusões são geradas a partir dos resultados reais. A inspeção qualitativa das tabelas, imagens e exemplos continua necessária, pois quantidade e tamanho não medem sozinhos a qualidade semântica.


In [9]:
stats = {int(row.test_id): row for _, row in estatisticas.iterrows()}
mais = estatisticas.loc[estatisticas.num_chunks.idxmax()]
menos = estatisticas.loc[estatisticas.num_chunks.idxmin()]
aud = df_auditoria.sum(numeric_only=True)
vector_map = {row["test_id"]: row for row in vector_quality_checks}

linhas_tabela = []
for test_id in range(1, 11):
    row = stats[test_id]
    cfg = CONFIGS[test_id]
    configuracao = (
        f'{cfg["chunk_size"]}, overlap {cfg["chunk_overlap"]}'
        if cfg["chunk_size"] else cfg["strategy"]
    )
    linhas_tabela.append(
        f'| {test_id} | {row.strategy} | {configuracao} | {int(row.num_chunks)} | '
        f'{row.avg_chunk_size:.1f} | {int(row.min_chunk_size)} | {int(row.max_chunk_size)} | '
        f'{vector_map[test_id]["avg_adjacent_cosine"]:.3f} |'
    )

exemplos = []
for test_id in range(1, 11):
    row = df_avaliacao[df_avaliacao.test_id == test_id].iloc[0]
    trecho = " ".join(row.text.split())[:300].replace("|", "\\|")
    exemplos.append(f'- **Teste {test_id} — {row.strategy}:** “{trecho}…”')

relatorio = f"""# Relatório — avaliação de estratégias de chunking

## Metodologia e economia de tokens

Os 10 testes foram executados em `bioetica_e_ia.md`, `escrita_academica_ia.md` e `twitter_algoritmo.md`. O teste vencedor foi aplicado aos outros nove documentos, totalizando 12 Markdown processados. A autora confirmou que os nove PDFs locais correspondem aos arquivos do Google Drive.

Essa abordagem foi adotada porque uma tentativa anterior de repetir os dez testes em todos os documentos excedeu o limite de tokens antes da conclusão. Para economizar tokens e viabilizar a atividade, todas as estratégias foram comparadas sobre a mesma amostra e com o mesmo modelo `{EMBEDDING_MODEL}`. Os embeddings finais foram produzidos localmente, sem chamadas ao OpenRouter.

## Configurações e estatísticas

| Teste | Estratégia | Configuração | Chunks | Média | Mínimo | Máximo | Cosseno adjacente |
|---:|---|---|---:|---:|---:|---:|---:|
{chr(10).join(linhas_tabela)}

Foram identificadas {int(aud.tabelas_markdown)} tabelas Markdown, {int(aud.referencias_imagens)} referências de imagem e {int(aud.marcadores_imagem)} marcadores `<!-- image -->`. A similaridade cosseno adjacente é apenas uma verificação complementar: overlap tende a aumentar essa medida e, por isso, ela não foi usada isoladamente.

## Exemplos de chunks

{chr(10).join(exemplos)}

## Análise obrigatória

### 1. Qual estratégia gerou mais chunks?

O teste {int(mais.test_id)}, `{mais.strategy}`, gerou {int(mais.num_chunks)} chunks. O tamanho reduzido aumenta fragmentação, armazenamento e quantidade de vetores.

### 2. Qual gerou menos chunks?

O teste {int(menos.test_id)}, `{menos.strategy}`, gerou {int(menos.num_chunks)} chunks. A baixa quantidade não significa maior qualidade, pois seções extensas podem concentrar assuntos diferentes.

### 3. Como o tamanho dos chunks variou?

Os testes fixos ficaram próximos de 200, 500, 1.000 e 2.000 caracteres. Parágrafos e sentenças variaram naturalmente. Recursive teve média de {stats[9].avg_chunk_size:.1f} e máximo de {int(stats[9].max_chunk_size)}; Markdown chegou a {int(stats[10].max_chunk_size)} caracteres.

### 4. Qual estratégia preservou melhor a estrutura?

Markdown preservou melhor headings e seções nos metadados. Recursive apresentou o melhor equilíbrio prático entre estrutura natural e tamanho controlado.

### 5. Como tabelas foram tratadas?

As tabelas simples foram mantidas como sintaxe Markdown. Cortes fixos podem dividir linhas e conceitos; Markdown e Recursive tendem a respeitar melhor os blocos, embora uma tabela maior que o limite ainda possa ser dividida. A conversão não garante preservação de células mescladas e layouts complexos.

### 6. Como imagens foram tratadas?

Foram contabilizadas referências Markdown e marcadores `<!-- image -->`. Quando existe apenas o marcador, a posição aproximada é preservada, mas os pixels e o significado visual não entram no embedding. Imagens não foram armazenadas separadamente por este pipeline.

### 7. Quais informações foram perdidas em PDF → Markdown?

Podem ter sido perdidos layout, colunas, posição exata, cores, células mescladas, fórmulas complexas, gráficos e relações entre figura e legenda. Não foi preservado um mapeamento confiável de página por chunk.

### 8. O corte por caracteres fragmentou conceitos?

Sim, especialmente com 200 caracteres. Ele pode interromper frases, referências e tabelas. Tamanhos maiores reduzem, mas não eliminam o problema; overlap recupera contexto ao custo de duplicação.

### 9. Parágrafos produziram chunks grandes?

Sim. O maior chunk do teste 7 teve {int(stats[7].max_chunk_size)} caracteres. A estratégia respeita a unidade natural, mas produz tamanhos irregulares.

### 10. Três sentenças preservaram melhor o contexto?

Preservaram melhor fronteiras linguísticas do que cortes fixos pequenos, mas três sentenças podem separar uma explicação longa ou juntar assuntos diferentes.

### 11. Recursive apresentou vantagens?

Sim. Ele tenta parágrafos, linhas, sentenças, espaços e caracteres, mantendo o máximo abaixo de 1.000 e overlap moderado de 10%.

### 12. Markdown preservou a estrutura semântica?

Sim. Os headings aparecem nos metadados, mas algumas seções ficaram extensas. Uma abordagem híbrida Markdown + Recursive seria mais segura.

### 13. Qual estratégia é mais adequada para RAG?

O teste 9, `{MELHOR_ESTRATEGIA}`, foi o mais adequado por equilibrar tamanho, contexto, hierarquia natural, overlap e consistência vetorial. A escolha não se baseou apenas no número de chunks.

### 14. Quais estratégias devem ser descartadas?

Como padrão, devem ser evitados 200 caracteres, 2.000 caracteres e overlap de 40%. Parágrafo puro e Markdown puro também exigem subdivisão quando geram unidades excessivas.

### 15. Quais estratégias usar nos próximos experimentos?

Recursive 1.000/100 e uma abordagem híbrida Markdown + Recursive. Uma avaliação futura deve usar perguntas reais e métricas como Recall@k ou MRR.

## Conclusão

O Recursive 1.000/100 produziu a melhor representação operacional para RAG nesta base. Todos os chunks receberam embeddings normalizados de 384 dimensões. Chunks maiores que o contexto do modelo foram processados em janelas e agregados, evitando descartar o final do texto. A conclusão ainda é exploratória porque não existe um conjunto rotulado de perguntas e respostas.
"""

(RESULTS_DIR / "RELATORIO.md").write_text(relatorio, encoding="utf-8")
print(relatorio)


# Relatório — avaliação de estratégias de chunking

## Metodologia e economia de tokens

Os 10 testes foram executados em `bioetica_e_ia.md`, `escrita_academica_ia.md` e `twitter_algoritmo.md`. O teste vencedor foi aplicado aos outros nove documentos, totalizando 12 Markdown processados. A autora confirmou que os nove PDFs locais correspondem aos arquivos do Google Drive.

Essa abordagem foi adotada porque uma tentativa anterior de repetir os dez testes em todos os documentos excedeu o limite de tokens antes da conclusão. Para economizar tokens e viabilizar a atividade, todas as estratégias foram comparadas sobre a mesma amostra e com o mesmo modelo `sentence-transformers/all-MiniLM-L6-v2`. Os embeddings finais foram produzidos localmente, sem chamadas ao OpenRouter.

## Configurações e estatísticas

| Teste | Estratégia | Configuração | Chunks | Média | Mínimo | Máximo | Cosseno adjacente |
|---:|---|---|---:|---:|---:|---:|---:|
| 1 | fixed_200 | 200, overlap 0 | 742 | 198.4 | 13 | 2

## 7. Exemplos para inspeção qualitativa

Execute a célula abaixo para comparar o primeiro chunk de cada estratégia sem imprimir embeddings.


In [10]:
for test_id in range(1, 11):
    exemplo = df_avaliacao[df_avaliacao.test_id == test_id].iloc[0]
    print("=" * 80)
    print(f"Teste {test_id}: {exemplo.strategy} | {exemplo.char_count} caracteres")
    print("Metadados:", exemplo.metadata)
    print(exemplo.text[:500].replace("\n", " "))


Teste 1: fixed_200 | 200 caracteres
Metadados: {}
273  <!-- image -->  ## Entre o algoritmo e o Juramento de Hipócrates: bioética na era da inteligência artificial  Juracy Barbosa dos Santos 1 , Guilhermina Rego 1 , Rui Nunes 1  1. Faculdade de Medic
Teste 2: fixed_500 | 500 caracteres
Metadados: {}
273  <!-- image -->  ## Entre o algoritmo e o Juramento de Hipócrates: bioética na era da inteligência artificial  Juracy Barbosa dos Santos 1 , Guilhermina Rego 1 , Rui Nunes 1  1. Faculdade de Medicina da Universidade do Porto, Porto, Portugal.  ## Resumo  O avanço da inteligência artificial tem transformado profundamente a prática médica. De sistemas de apoio à decisão clínica a algoritmos de triagem e diagnóstico, a inteligência artificial tem demons -trado potencial para diagnósticos precoc
Teste 3: fixed_1000 | 1000 caracteres
Metadados: {}
273  <!-- image -->  ## Entre o algoritmo e o Juramento de Hipócrates: bioética na era da inteligência artificial  Juracy Barbosa dos Santos 1 , G

# Relatório — avaliação de estratégias de chunking

## Metodologia e economia de tokens

Os 10 testes foram executados em `bioetica_e_ia.md`, `escrita_academica_ia.md` e `twitter_algoritmo.md`. O teste vencedor foi aplicado aos outros nove documentos, totalizando 12 Markdown processados. A autora confirmou que os nove PDFs locais correspondem aos arquivos do Google Drive.

Essa abordagem foi adotada porque uma tentativa anterior de repetir os dez testes em todos os documentos excedeu o limite de tokens antes da conclusão. Para economizar tokens e viabilizar a atividade, todas as estratégias foram comparadas sobre a mesma amostra e com o mesmo modelo `sentence-transformers/all-MiniLM-L6-v2`. Os embeddings finais foram produzidos localmente, sem chamadas ao OpenRouter.

## Configurações e estatísticas

| Teste | Estratégia | Configuração | Chunks | Média | Mínimo | Máximo | Cosseno adjacente |
|---:|---|---|---:|---:|---:|---:|---:|
| 1 | fixed_200 | 200, overlap 0 | 742 | 198.4 | 13 | 200 | 0.466 |
| 2 | fixed_500 | 500, overlap 0 | 298 | 494.6 | 63 | 500 | 0.569 |
| 3 | fixed_1000 | 1000, overlap 0 | 150 | 984.6 | 213 | 1000 | 0.682 |
| 4 | fixed_2000 | 2000, overlap 0 | 76 | 1951.3 | 440 | 2000 | 0.786 |
| 5 | fixed_500_overlap_50 | 500, overlap 50 | 330 | 496.3 | 156 | 500 | 0.620 |
| 6 | fixed_500_overlap_200 | 500, overlap 200 | 494 | 497.0 | 156 | 500 | 0.730 |
| 7 | paragraph | paragraph | 447 | 329.9 | 1 | 4291 | 0.404 |
| 8 | three_sentences | three_sentences | 378 | 389.6 | 5 | 1668 | 0.473 |
| 9 | recursive_1000_overlap_100 | 1000, overlap 100 | 211 | 708.8 | 65 | 996 | 0.604 |
| 10 | markdown_headers | markdown_headers | 67 | 2217.2 | 10 | 9877 | 0.570 |

Foram identificadas 144 tabelas Markdown, 0 referências de imagem e 171 marcadores `<!-- image -->`. A similaridade cosseno adjacente é apenas uma verificação complementar: overlap tende a aumentar essa medida e, por isso, ela não foi usada isoladamente.

## Exemplos de chunks

- **Teste 1 — fixed_200:** “273 <!-- image --> ## Entre o algoritmo e o Juramento de Hipócrates: bioética na era da inteligência artificial Juracy Barbosa dos Santos 1 , Guilhermina Rego 1 , Rui Nunes 1 1. Faculdade de Medic…”
- **Teste 2 — fixed_500:** “273 <!-- image --> ## Entre o algoritmo e o Juramento de Hipócrates: bioética na era da inteligência artificial Juracy Barbosa dos Santos 1 , Guilhermina Rego 1 , Rui Nunes 1 1. Faculdade de Medicina da Universidade do Porto, Porto, Portugal. ## Resumo O avanço da inteligência artificial tem transfo…”
- **Teste 3 — fixed_1000:** “273 <!-- image --> ## Entre o algoritmo e o Juramento de Hipócrates: bioética na era da inteligência artificial Juracy Barbosa dos Santos 1 , Guilhermina Rego 1 , Rui Nunes 1 1. Faculdade de Medicina da Universidade do Porto, Porto, Portugal. ## Resumo O avanço da inteligência artificial tem transfo…”
- **Teste 4 — fixed_2000:** “273 <!-- image --> ## Entre o algoritmo e o Juramento de Hipócrates: bioética na era da inteligência artificial Juracy Barbosa dos Santos 1 , Guilhermina Rego 1 , Rui Nunes 1 1. Faculdade de Medicina da Universidade do Porto, Porto, Portugal. ## Resumo O avanço da inteligência artificial tem transfo…”
- **Teste 5 — fixed_500_overlap_50:** “273 <!-- image --> ## Entre o algoritmo e o Juramento de Hipócrates: bioética na era da inteligência artificial Juracy Barbosa dos Santos 1 , Guilhermina Rego 1 , Rui Nunes 1 1. Faculdade de Medicina da Universidade do Porto, Porto, Portugal. ## Resumo O avanço da inteligência artificial tem transfo…”
- **Teste 6 — fixed_500_overlap_200:** “273 <!-- image --> ## Entre o algoritmo e o Juramento de Hipócrates: bioética na era da inteligência artificial Juracy Barbosa dos Santos 1 , Guilhermina Rego 1 , Rui Nunes 1 1. Faculdade de Medicina da Universidade do Porto, Porto, Portugal. ## Resumo O avanço da inteligência artificial tem transfo…”
- **Teste 7 — paragraph:** “273…”
- **Teste 8 — three_sentences:** “273 <!-- image --> ## Entre o algoritmo e o Juramento de Hipócrates: bioética na era da inteligência artificial Juracy Barbosa dos Santos 1 , Guilhermina Rego 1 , Rui Nunes 1 1. Faculdade de Medicina da Universidade do Porto, Porto, Portugal. ## Resumo O avanço da inteligência artificial tem transfo…”
- **Teste 9 — recursive_1000_overlap_100:** “273 <!-- image --> ## Entre o algoritmo e o Juramento de Hipócrates: bioética na era da inteligência artificial Juracy Barbosa dos Santos 1 , Guilhermina Rego 1 , Rui Nunes 1 1. Faculdade de Medicina da Universidade do Porto, Porto, Portugal. ## Resumo…”
- **Teste 10 — markdown_headers:** “273 <!-- image -->…”

## Análise obrigatória

### 1. Qual estratégia gerou mais chunks?

O teste 1, `fixed_200`, gerou 742 chunks. O tamanho reduzido aumenta fragmentação, armazenamento e quantidade de vetores.

### 2. Qual gerou menos chunks?

O teste 10, `markdown_headers`, gerou 67 chunks. A baixa quantidade não significa maior qualidade, pois seções extensas podem concentrar assuntos diferentes.

### 3. Como o tamanho dos chunks variou?

Os testes fixos ficaram próximos de 200, 500, 1.000 e 2.000 caracteres. Parágrafos e sentenças variaram naturalmente. Recursive teve média de 708.8 e máximo de 996; Markdown chegou a 9877 caracteres.

### 4. Qual estratégia preservou melhor a estrutura?

Markdown preservou melhor headings e seções nos metadados. Recursive apresentou o melhor equilíbrio prático entre estrutura natural e tamanho controlado.

### 5. Como tabelas foram tratadas?

As tabelas simples foram mantidas como sintaxe Markdown. Cortes fixos podem dividir linhas e conceitos; Markdown e Recursive tendem a respeitar melhor os blocos, embora uma tabela maior que o limite ainda possa ser dividida. A conversão não garante preservação de células mescladas e layouts complexos.

### 6. Como imagens foram tratadas?

Foram contabilizadas referências Markdown e marcadores `<!-- image -->`. Quando existe apenas o marcador, a posição aproximada é preservada, mas os pixels e o significado visual não entram no embedding. Imagens não foram armazenadas separadamente por este pipeline.

### 7. Quais informações foram perdidas em PDF → Markdown?

Podem ter sido perdidos layout, colunas, posição exata, cores, células mescladas, fórmulas complexas, gráficos e relações entre figura e legenda. Não foi preservado um mapeamento confiável de página por chunk.

### 8. O corte por caracteres fragmentou conceitos?

Sim, especialmente com 200 caracteres. Ele pode interromper frases, referências e tabelas. Tamanhos maiores reduzem, mas não eliminam o problema; overlap recupera contexto ao custo de duplicação.

### 9. Parágrafos produziram chunks grandes?

Sim. O maior chunk do teste 7 teve 4291 caracteres. A estratégia respeita a unidade natural, mas produz tamanhos irregulares.

### 10. Três sentenças preservaram melhor o contexto?

Preservaram melhor fronteiras linguísticas do que cortes fixos pequenos, mas três sentenças podem separar uma explicação longa ou juntar assuntos diferentes.

### 11. Recursive apresentou vantagens?

Sim. Ele tenta parágrafos, linhas, sentenças, espaços e caracteres, mantendo o máximo abaixo de 1.000 e overlap moderado de 10%.

### 12. Markdown preservou a estrutura semântica?

Sim. Os headings aparecem nos metadados, mas algumas seções ficaram extensas. Uma abordagem híbrida Markdown + Recursive seria mais segura.

### 13. Qual estratégia é mais adequada para RAG?

O teste 9, `recursive_1000_overlap_100`, foi o mais adequado por equilibrar tamanho, contexto, hierarquia natural, overlap e consistência vetorial. A escolha não se baseou apenas no número de chunks.

### 14. Quais estratégias devem ser descartadas?

Como padrão, devem ser evitados 200 caracteres, 2.000 caracteres e overlap de 40%. Parágrafo puro e Markdown puro também exigem subdivisão quando geram unidades excessivas.

### 15. Quais estratégias usar nos próximos experimentos?

Recursive 1.000/100 e uma abordagem híbrida Markdown + Recursive. Uma avaliação futura deve usar perguntas reais e métricas como Recall@k ou MRR.

## Conclusão

O Recursive 1.000/100 produziu a melhor representação operacional para RAG nesta base. Todos os chunks receberam embeddings normalizados de 384 dimensões. Chunks maiores que o contexto do modelo foram processados em janelas e agregados, evitando descartar o final do texto. A conclusão ainda é exploratória porque não existe um conjunto rotulado de perguntas e respostas.
